In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

# Training Object Detection Model - YOLO26 Nano

In [ ]:
# download the dataset from Roboflow

from roboflow import Roboflow

api_key = os.getenv("ROBOFLOW_API_KEY")

rf = Roboflow(api_key=api_key)
project = rf.workspace("david-van-kriedt-s-workspace").project("offsider")
version = project.version(2)
dataset = version.download("yolov8", location="../data/sign-detection")

In [ ]:
# Training YOLO26

from ultralytics import YOLO

model = YOLO("yolo26n.pt")

results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=30,
    imgsz=640,
    batch=16,
    plots=True,
    project="../runs/detect",
    name="offsider_baseline"
)

In [ ]:
# Save the best model weights to the models directory

import shutil
from pathlib import Path

models_dir = Path("../models")
models_dir.mkdir(parents=True, exist_ok=True)

best_weights = Path(results.save_dir) / "weights" / "best.pt"

target_weights = models_dir / "best.pt"
shutil.copy(best_weights, target_weights)

print(f"Model saved to: {target_weights.resolve()}")

In [ ]:
# Test the model on the test dataset

test_model = YOLO("../models/best.pt")

test_results = test_model.predict(source="../data/sign-detection/test/images", conf=0.8, max_det=3)

test_results[0].show()